In [1]:
# charger le fichier brut
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

df_brut = pd.read_csv("/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/01_Donnees_brutes/orders_export_1.csv")
print(f"Nombre de lignes brutes : {len(df_brut)}")
df_brut.head()

Nombre de lignes brutes : 58


,Name,Email,Financial Status,Paid at,Fulfillment Status,Fulfilled at,Accepts Marketing,Currency,Subtotal,Shipping,...,Tax 5 Value,Phone,Receipt Number,Duties,Billing Province Name,Shipping Province Name,Payment ID,Payment Terms Name,Next Payment Due At,Payment References
0,#1057,arnaud.herbin59@gmail.com,paid,2026-06-14 18:54:34 +0200,unfulfilled,NaN,no,EUR,36.0,16.60,...,NaN,NaN,NaN,NaN,NaN,NaN,rFFTCqc6WB8AAaE2iMSHPKm5U,NaN,NaN,rFFTCqc6WB8AAaE2iMSHPKm5U
1,#1056,marion_pierre@hotmail.fr,paid,2026-06-11 10:52:49 +0200,unfulfilled,NaN,no,EUR,526.0,43.80,...,NaN,NaN,NaN,NaN,NaN,NaN,rZke3jVECoUiHZ6gTwwzaVAWh,NaN,NaN,rZke3jVECoUiHZ6gTwwzaVAWh
2,#1055,monstre-plante@live.fr,refunded,2026-06-10 09:37:31 +0200,unfulfilled,NaN,no,EUR,185.0,21.36,...,NaN,NaN,NaN,NaN,NaN,NaN,rLmWTBdyTmTWr0ajpxmj0z4X2,NaN,NaN,rLmWTBdyTmTWr0ajpxmj0z4X2 + z3NvYdoSJgF17LzkBN...
3,#1054,enzo.viard@orange.fr,refunded,2026-06-02 20:57:21 +0200,unfulfilled,NaN,no,EUR,653.0,37.80,...,NaN,NaN,NaN,NaN,NaN,NaN,rQQ9Ld8zG9ntU3AbgGIKWgB75,NaN,NaN,rQQ9Ld8zG9ntU3AbgGIKWgB75 + znOLxJUleFJPAUI3Jq...
4,#1053,vincentcontant@live.fr,refunded,2026-06-01 11:13:47 +0200,unfulfilled,NaN,no,EUR,175.0,25.33,...,NaN,NaN,NaN,NaN,NaN,NaN,rhNylNsAqcpIWQlfVfXo2FBFk,NaN,NaN,rhNylNsAqcpIWQlfVfXo2FBFk + zzGVfLQiEBQXY2jDg0...


In [2]:
#sélectionner les colonnes utiles 
colonnes_utiles = ["Name", "Total", "Financial Status", "Created at", "Lineitem name", "Lineitem quantity", "Discount Code", "Refunded Amount"]
df_select = df_brut[colonnes_utiles].copy()
print(df_select.shape)
df_select.head()

(58, 8)


,Name,Total,Financial Status,Created at,Lineitem name,Lineitem quantity,Discount Code,Refunded Amount
0,#1057,52.60,paid,2026-06-14 18:54:33 +0200,Grille-pain BOSCH TAT6A004EM,1,NaN,0.00
1,#1056,569.80,paid,2026-06-11 10:52:48 +0200,"Réfrigérateur TOP LOADED , Blanc, Compresseur ...",1,NaN,0.00
2,#1055,206.36,refunded,2026-06-10 09:37:31 +0200,Toaster 2 tranches Smeg TSF01UJEU-Union Jack,1,NaN,206.36
3,#1054,690.80,refunded,2026-06-02 20:57:20 +0200,Dometic CFX5 35 – Réfrigérateur portable à com...,1,NaN,690.80
4,#1053,200.33,refunded,2026-06-01 11:13:46 +0200,Annexe gonflable Dometic Ascension FTX TC Annexe,1,NaN,200.33


In [3]:
# nettoyer la colonne de date et créer la colonne Mois
df_select["Created at"] = pd.to_datetime(df_select["Created at"], utc=True)
df_select["Mois"] = df_select["Created at"].dt.strftime("%Y-%m")
df_select[["Name", "Created at", "Mois"]].head()

,Name,Created at,Mois
0,#1057,2026-06-14 16:54:33+00:00,2026-06
1,#1056,2026-06-11 08:52:48+00:00,2026-06
2,#1055,2026-06-10 07:37:31+00:00,2026-06
3,#1054,2026-06-02 18:57:20+00:00,2026-06
4,#1053,2026-06-01 09:13:46+00:00,2026-06


In [4]:
#appliquer le filtre des commandes test (numéro < 1035)
df_select["Numero"] = df_select["Name"].str.replace("#", "").astype(int)

df_propre = df_select[df_select["Numero"] >= 1035].copy()

print(f"Lignes avant filtre : {len(df_select)}")
print(f"Lignes après filtre (commandes test retirées) : {len(df_propre)}")
print(f"Nombre de commandes test retirées : {len(df_select) - len(df_propre)}")

Lignes avant filtre : 58
Lignes après filtre (commandes test retirées) : 23
Nombre de commandes test retirées : 35


In [5]:
#créer la colonne "Statut simplifié"
print(df_propre["Financial Status"].value_counts(dropna=False))
print()

traduction_statut = {
    "paid": "Payee",
    "refunded": "Remboursee",
    "partially_refunded": "Partiellement remboursee"
}

df_propre["Statut simplifie"] = df_propre["Financial Status"].map(traduction_statut)

print(df_propre["Statut simplifie"].value_counts())
df_propre[["Name", "Total", "Financial Status", "Statut simplifie"]].head()

Financial Status
paid                  13
refunded               9
partially_refunded     1
Name: count, dtype: int64

Statut simplifie
Payee                       13
Remboursee                   9
Partiellement remboursee     1
Name: count, dtype: int64


,Name,Total,Financial Status,Statut simplifie
0,#1057,52.60,paid,Payee
1,#1056,569.80,paid,Payee
2,#1055,206.36,refunded,Remboursee
3,#1054,690.80,refunded,Remboursee
4,#1053,200.33,refunded,Remboursee


In [6]:
#sélectionner les colonnes finales et corriger le fuseau horaire
df_final = df_propre[["Name", "Total", "Statut simplifie", "Mois", "Lineitem name", "Lineitem quantity", "Created at", "Refunded Amount"]].copy()
df_final = df_final.rename(columns={"Lineitem name": "Produit", "Lineitem quantity": "Quantite"})
df_final["Created at"] = df_final["Created at"].dt.tz_localize(None)

print(df_final.shape)
df_final.head()

(23, 8)


,Name,Total,Statut simplifie,Mois,Produit,Quantite,Created at,Refunded Amount
0,#1057,52.60,Payee,2026-06,Grille-pain BOSCH TAT6A004EM,1,2026-06-14 16:54:33,0.00
1,#1056,569.80,Payee,2026-06,"Réfrigérateur TOP LOADED , Blanc, Compresseur ...",1,2026-06-11 08:52:48,0.00
2,#1055,206.36,Remboursee,2026-06,Toaster 2 tranches Smeg TSF01UJEU-Union Jack,1,2026-06-10 07:37:31,206.36
3,#1054,690.80,Remboursee,2026-06,Dometic CFX5 35 – Réfrigérateur portable à com...,1,2026-06-02 18:57:20,690.80
4,#1053,200.33,Remboursee,2026-06,Annexe gonflable Dometic Ascension FTX TC Annexe,1,2026-06-01 09:13:46,200.33


In [7]:
#sauvegarder le fichier propre final
chemin_sortie = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres/commandes_propres_v3.xlsx"
df_final.to_excel(chemin_sortie, index=False)
print(f"Fichier sauvegardé : {chemin_sortie}")
print(f"Nombre de lignes sauvegardées : {len(df_final)}")

Fichier sauvegardé : /Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres/commandes_propres_v3.xlsx
Nombre de lignes sauvegardées : 23
